## Choose a large transformer model and try to fine-tune it on the dataset of your choice using LoRA.

In [11]:
from datasets import load_dataset, Dataset, DatasetDict
from transformers import AutoTokenizer,  AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from peft import LoraConfig, get_peft_model
import torch
import numpy as np

In [12]:
news_ds = load_dataset("mdonigian/iab-news-classification")
news_ds = news_ds.rename_column("iab_category", "labels")
news_ds = news_ds.class_encode_column("labels")

In [13]:
news_ds = news_ds['train'].train_test_split(test_size=0.1)

In [14]:
news_ds

DatasetDict({
    train: Dataset({
        features: ['url', 'domain', 'maintext', 'labels'],
        num_rows: 95652
    })
    test: Dataset({
        features: ['url', 'domain', 'maintext', 'labels'],
        num_rows: 10628
    })
})

In [15]:
train_dataset = news_ds['train']

In [16]:
train_dataset

Dataset({
    features: ['url', 'domain', 'maintext', 'labels'],
    num_rows: 95652
})

In [17]:
train_dataset, validation_dataset = train_dataset.train_test_split(test_size=0.1).values()

In [18]:
train_dataset

Dataset({
    features: ['url', 'domain', 'maintext', 'labels'],
    num_rows: 86086
})

In [19]:
news_ds['train'] = train_dataset
news_ds['validation'] = validation_dataset
news_ds

DatasetDict({
    train: Dataset({
        features: ['url', 'domain', 'maintext', 'labels'],
        num_rows: 86086
    })
    test: Dataset({
        features: ['url', 'domain', 'maintext', 'labels'],
        num_rows: 10628
    })
    validation: Dataset({
        features: ['url', 'domain', 'maintext', 'labels'],
        num_rows: 9566
    })
})

In [20]:
news_ds['train']['labels'][:5]

[22, 9, 23, 29, 17]

In [21]:
model_id = "distilbert/distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=35, device_map='auto')

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [22]:
def tokenize(batch):
    return tokenizer(batch['maintext'], truncation=True, max_length=512)

news_ds= news_ds.map(tokenize, batched=True)

Map:   0%|          | 0/86086 [00:00<?, ? examples/s]

Map:   0%|          | 0/10628 [00:00<?, ? examples/s]

Map:   0%|          | 0/9566 [00:00<?, ? examples/s]

In [23]:
for name, module in model.named_modules():
    print(name, module)

 DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
          (ffn): FFN(
            (dropout): Dropout

In [25]:
lora_config = LoraConfig(r= 16,
                         lora_alpha = 32,
                         target_modules=['q_lin', 'v_lin'],
                         lora_dropout=0.1,
                         modules_to_save=['pre_classifier', "classifier"],
                         bias='none',
                         task_type="SEQ_CLS")

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 912,419 || all params: 67,892,806 || trainable%: 1.3439


In [26]:
def compute_metric(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy" : (preds == labels).mean()}

In [27]:
data_collator = DataCollatorWithPadding(tokenizer)

training_args = TrainingArguments(
    output_dir = 'lora_bert',
    num_train_epochs=3,
    gradient_accumulation_steps= 4,
    per_device_train_batch_size= 8,
    learning_rate=2e-4,
    logging_steps=250,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True
)

trainer = Trainer(
    model = model,
    args=training_args,
    train_dataset=news_ds['train'],
    eval_dataset=news_ds['validation'],
    processing_class = tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metric
)
trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy
1,2.108148,0.478006,0.831487
2,1.837457,0.434342,0.844658
3,1.620976,0.421891,0.845913


TrainOutput(global_step=8073, training_loss=2.075177807325356, metrics={'train_runtime': 2242.29, 'train_samples_per_second': 115.176, 'train_steps_per_second': 3.6, 'total_flos': 3.495466277461296e+16, 'train_loss': 2.075177807325356, 'epoch': 3.0})

In [28]:
trainer.predict(news_ds['test'])

PredictionOutput(predictions=array([[-4.631371  , -4.9405866 , -5.1809    , ..., -4.4169135 ,
        -5.804749  ,  5.6878138 ],
       [-3.561906  , -3.0028536 , -2.7390592 , ..., -4.6065917 ,
         7.034347  , -3.8840458 ],
       [-3.0706065 , -4.566841  , -5.815273  , ..., -2.0261045 ,
        -7.9394217 , -0.65685606],
       ...,
       [-7.4710994 , -7.170515  , -5.610246  , ..., -3.6332757 ,
        -9.586872  , -9.550737  ],
       [-2.5521247 , -6.1120515 , -5.5308375 , ..., -2.344363  ,
        -4.476482  , -7.100985  ],
       [-5.056686  , -5.881008  , -4.4816527 , ..., -2.744548  ,
        -5.3475895 , -6.7456594 ]], shape=(10628, 35), dtype=float32), label_ids=array([23, 33, 23, ..., 19, 20, 13], shape=(10628,)), metrics={'test_loss': 0.4363516569137573, 'test_accuracy': 0.8478547233722243, 'test_runtime': 40.0662, 'test_samples_per_second': 265.261, 'test_steps_per_second': 33.17})